# 5.12 — Separating Data Loading, Training, and Inference

This notebook demonstrates a **clean ML architecture** in this repo by keeping three responsibilities separate:

- **Data loading**: reads raw data only (no fitting, no splitting, no target derivation).
- **Training**: fits preprocessing + model, evaluates, and **saves artifacts**.
- **Inference**: **loads artifacts** and runs `transform()` + `predict()` only (never refits).

Key mental model: **Training produces artifacts. Inference consumes artifacts.**

## 1) Project structure (the important parts)

- `src/data_loader.py` → raw CSV loading + validation
- `src/training_pipeline.py` / `src/train.py` → training orchestration (fitting happens here)
- `src/prediction_pipeline.py` → inference orchestration (no fitting)
- `models/` → persisted model + preprocessor
- `reports/` → evaluation + predictions
- `logs/` → experiment log

In [ ]:
from pathlib import Path

root = Path('.').resolve()
print('Repo root:', root)

for p in ['src', 'data/raw', 'models', 'reports', 'logs']:
    path = root / p
    print(f'- {p}:', 'OK' if path.exists() else 'MISSING')

print('\nKey src modules:')
for name in ['data_loader.py', 'training_pipeline.py', 'train.py', 'prediction_pipeline.py']:
    print(' -', name)

## 2) Data loading layer (raw-only)

The loader reads the raw CSV and returns a dataframe.
It does **not** split, scale, engineer features, or create a target label.

In [ ]:
from src.data_loader import load_csv

df_raw = load_csv('data/raw/source_demo_crops_20260321.csv')
df_raw.head()

## 3) Training layer (fitting + saving artifacts)

Training is the only place where fitting happens:

- train/test split
- fit preprocessor on train
- train model
- evaluate
- save artifacts to `models/`
- write evaluation to `reports/` and append `logs/experiment_log.csv`

For the demo CSV, training derives a **binary target** from `yield_kg` (median split).

In [ ]:
from src.training_pipeline import run_training_pipeline

metrics = run_training_pipeline(data_path='data/raw/source_demo_crops_20260321.csv')
metrics

In [ ]:
from src.config import DEFAULT_MODEL_PATH, DEFAULT_PREPROCESSOR_PATH, DEFAULT_EVALUATION_REPORT_PATH, DEFAULT_EXPERIMENT_LOG_PATH

paths = [DEFAULT_MODEL_PATH, DEFAULT_PREPROCESSOR_PATH, DEFAULT_EVALUATION_REPORT_PATH, DEFAULT_EXPERIMENT_LOG_PATH]
for p in paths:
    p = Path(p)
    print(str(p), '->', 'exists' if p.exists() else 'missing')

## 4) Inference layer (load artifacts + transform + predict)

Inference must never call `fit()` or `fit_transform()`.
It loads the saved preprocessor/model artifacts and runs `transform()` + `predict()`.

We pass `target_column='yield_kg'` so the inference loader drops it if present (avoid leakage).

In [ ]:
from src.prediction_pipeline import run_prediction_pipeline

preds = run_prediction_pipeline(
    data_path='data/raw/source_demo_crops_20260321.csv',
    target_column='yield_kg',
    output_path='reports/predictions.csv',
)
preds.head()